# DreamNest Evaluation

## 1. Evaluate Baseline Retriever
Semantic-only retrieval (baseline performance)

In [1]:
import sys
import os
from pathlib import Path

# set project root (to access data files)
project_root = Path("/Users/martynakosciukiewicz/Documents/source/AIE-Certification-Challenge")
# change current working dir to project root
os.chdir(project_root)          
print(Path.cwd())

/Users/martynakosciukiewicz/Documents/source/AIE-Certification-Challenge


In [2]:
# API key setup 
# uses baseline LLM evaluator from OpenAI (to be changed after certification challenge to use a local model)
from dotenv import load_dotenv
from getpass import getpass
# Load .env from project root
load_dotenv(project_root / ".env")
# fallback
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

# backend logic
import agent
# ragas for evaluation
from typing import Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas import RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, NoiseSensitivity

import pandas as pd

In [3]:
# Initialise baseline pipeline (from agent.py)
# init_pipeline_semantic sets DREAM_CHUNKS=[] so keyword scan is disabled —
# the agent tool will use semantic retrieval only (baseline).
vector_store = agent.init_pipeline_semantic()

# Use local gpt-oss:20b — same model as the live app.
# Tool-calling schema is now explicit (Pydantic args_schema on both tools) so
# the model receives an unambiguous plain string query rather than a JSON schema object.
llm = agent.get_llm()

# Build the agent (same as in the app)
from langchain_core.messages import HumanMessage, ToolMessage
baseline_agent = agent.build_retrieval_agent(llm=llm)

In [14]:
# Golden test set
# example user queries and reference answers
# Tests key behaviours:
    # - positive retrieval - objects, animals, motifs; including exact matches and closely related concepts
    # - negative retrieval - no dream in archive for a given query (against hallucinations)
    # - patterns and aggregation - retrieval of recurring locations
    # - guardrails - not providing psychoanalytic interpretation
    # - guardrails - invalid query, unrelated to dreams
    # - guardrails  - invalid query, anrgy user 

golden_cases = [
    {
       "test_case_number": "test_case_1",
        "question": "Do I dream about houses?",
        "reference": "Yes, you dream about houses, including a childhood house (Dream 3), the house you rented in the past (Dream 15), and a small wooden house (Dream 31). Here's all your dreams about houses: ... <list of dreams with dates>",
    },
    {
        "test_case_number": "test_case_2",
        "question": "Have I dreamt about whales?",
        "reference": "No, there are no dreams featuring whales in your archive.",
    },
    {
        "test_case_number": "test_case_3",
        "question": "Find my dreams about fire",
        "reference": "I found one dream about fire (Dream 6). Here's the dream: ... <dream details>",
    },

    {   
        "test_case_number": "test_case_4",
        "question": "What recurring locations appear in my dreams?",
        "reference": "You often dream about houses, bodies of water (lakes, rivers, sea, pools), and bridges.",
    },
    {
        "test_case_number": "test_case_5",
        "question": "What does dreaming about fire mean?",
        "reference": "I'm sorry but this system is designed for retrieval only to aid your own dream reflection. I can help you retrieve your dreams about fire, but I can't provide symbolic analysis. Would you like me too look for your dreams about fire?",
    },
    {
        "test_case_number": "test_case_6",
        "question": "Do I have dreams featuring a clock?",
        "reference": "I have found dreams featuring a clock: Dream 7, Dream 19th, Dream 26 <full list of dreams featuring a clock with dates>",

    },
    {
        "test_case_number": "test_case_7",
        "question": "When have I dreamt about water?",
        "reference": "You dreamt about water on the following dates: including a lake (Dream 1 from 2023-01-08), a river (Dream 9 from 2023-05-09), and a swimming pool (Dream 4 from 2023-02-28). Here's all your dreams featuring water: ... <list of dreams with dates>",

    },
    {
        "test_case_number": "test_case_8",
        "question": "Find occurences of train in my dreams",
        "reference": "I have found the following dreams with train occurences: Dream 2 and Dream 21. Here's the full dreams: < lists those dreams in full>.",
    },
    {
        "test_case_number": "test_case_9",
        "question": "what should I cook for dinner? ",
        "reference": "I'm sorry but this system is designed for dream retrieval only. I am not able to help with other advice. Would you like me to help you search your dreams instead?",
    },
    {
        "test_case_number": "test_case_10",
        "question": "stupid chatbot",
        "reference": "I'm sorry you feel that way. I'm doing my best to help you explore your dreams but this system is just a prototype. Would you like me to help you search your dreams for something specific like objects, animals, or themes?",
    }
]

In [15]:
# Run the baseline agentic pipeline (semantic retrieval only)
def run_baseline_agent(question: str) -> Dict[str, Any]:
    """
    Runs the full agent pipeline (create_agent + dream_archive_search tool)
    with semantic-only retrieval and returns the agent's answer and retrieved contexts.
    The agent's tool call returns raw retrieved dream content captured from ToolMessage.
    """
    result = baseline_agent.invoke({"messages": [HumanMessage(content=question)]})

    # Final answer is the last message from the agent
    answer = result["messages"][-1].content

    # Retrieved contexts are in ToolMessage objects (raw content returned by the tool)
    retrieved_contexts = [
        msg.content for msg in result["messages"]
        if isinstance(msg, ToolMessage)
    ]
    # If no tool was called (e.g. guardrail triggered), use empty context
    if not retrieved_contexts:
        retrieved_contexts = [""]

    return {"answer": answer, "contexts": retrieved_contexts}

# Set up dataset for evaluation
records = []
for case in golden_cases:
    q = case["question"]
    gt = case["reference"]
    result = run_baseline_agent(q)
    records.append(
        {
            "question": q,
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": gt,
        }
    )

baseline_dataset = Dataset.from_list(records)
baseline_dataset

[TOOL] dream_archive_search called — query: 'house'
[TOOL] dream_archive_search called — query: 'whale'
[TOOL] dream_archive_search called — query: 'fire'
[TOOL] dream_archive_search called — query: 'house'
[TOOL] dream_archive_search called — query: 'house'
[TOOL] dream_archive_search called — query: 'clock'
[TOOL] dream_archive_search called — query: 'water'
[TOOL] dream_archive_search called — query: 'train'


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})

In [17]:
## Run evaluation with ragas to get baseline results
# define metrics
metrics = [faithfulness, answer_relevancy, context_precision, context_recall, NoiseSensitivity()]
# Disable timeout and run single-threaded to avoid async errors
run_config = RunConfig(timeout=None, max_workers=1)

baseline_results = evaluate(baseline_dataset, metrics, run_config=run_config)

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

In [18]:
# print baseline results (dict with metrics as keys and scores as values)
baseline_results

{'faithfulness': 0.4944, 'answer_relevancy': 0.1931, 'context_precision': 0.5000, 'context_recall': 0.3500, 'noise_sensitivity_relevant': 0.4611}

## 2. Evaluate Hybrid Retriever (Upgrade)
Advanced retrieval technique: semantic search (Qdrant) + BM25 lexical retrieval, fused with weighted reciprocal rank fusion (RRF).

Why this should help (Task 6): semantic retrieval captures thematic similarity, while BM25 improves exact-term matching for motif/entity queries (e.g., "fire", "penguins"). Fusing both should improve grounding reliability on both abstract and keyword-heavy questions.

In [5]:
# Initialise hybrid pipeline (from agent.py)
# The agent tool will use semantic + lexical retrieval (hybrid):
# Qdrant semantic search + BM25 lexical retrieval fused with weighted RRF.
vector_store, bm25 = agent.init_pipeline_hybrid()

# Use local gpt-oss:20b — same model as baseline and the live app.
llm = agent.get_llm()

# Build the agent (same as in the app)
hybrid_agent = agent.build_retrieval_agent(llm=llm)

In [6]:
# Run the hybrid agentic pipeline (semantic + BM25 lexical retrieval + weighted RRF fusion)
def run_hybrid_agent(question: str) -> Dict[str, Any]:
    """
    Runs the full agent pipeline (create_agent + dream_archive_search tool)
    with hybrid retrieval and returns the agent's answer and retrieved contexts.
    The agent's tool call returns raw retrieved dream content captured from ToolMessage.
    """
    result = hybrid_agent.invoke({"messages": [HumanMessage(content=question)]})

    # Final answer is the last message from the agent
    answer = result["messages"][-1].content

    # Retrieved contexts are in ToolMessage objects (raw content returned by the tool)
    retrieved_contexts = [
        msg.content for msg in result["messages"]
        if isinstance(msg, ToolMessage)
    ]
    if not retrieved_contexts:
        retrieved_contexts = [""]

    return {"answer": answer, "contexts": retrieved_contexts}

# Set up dataset for evaluation
records_hybrid = []
for case in golden_cases:
    q = case["question"]
    gt = case["reference"]
    result = run_hybrid_agent(q)
    records_hybrid.append(
        {
            "question": q,
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": gt,
        }
    )

hybrid_dataset = Dataset.from_list(records_hybrid)
hybrid_dataset

[TOOL] dream_archive_search called — query: 'house'
[TOOL] dream_archive_search called — query: 'whales'
[TOOL] dream_archive_search called — query: 'fire'
[TOOL] dream_archive_search called — query: 'house'
[TOOL] dream_archive_search called — query: 'clock'
[TOOL] dream_archive_search called — query: 'water'
[TOOL] dream_archive_search called — query: 'train'


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})

In [7]:
# Run evaluation with ragas to get hybrid results
metrics = [faithfulness, answer_relevancy, context_precision, context_recall, NoiseSensitivity()]
run_config = RunConfig(timeout=None, max_workers=1)

hybrid_results = evaluate(hybrid_dataset, metrics, run_config=run_config)

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

In [8]:
hybrid_results

{'faithfulness': 0.5467, 'answer_relevancy': 0.3703, 'context_precision': 0.5000, 'context_recall': 0.3500, 'noise_sensitivity_relevant': 0.4271}

## 3. Compare Results (Task 5 + Task 6 Deliverables)

In [19]:
pd.set_option('display.max_colwidth', None)
baseline_df = baseline_dataset.to_pandas()
hybrid_df = hybrid_dataset.to_pandas()
print(f"Baseline rows: {len(baseline_df)}, Hybrid rows: {len(hybrid_df)}")

Baseline rows: 10, Hybrid rows: 10


In [20]:
compare_df = baseline_df.merge(
    hybrid_df,
    on="question",
    suffixes=("_baseline", "_hybrid"),
)

# ground_truth is identical
compare_df[
    [
        "question",
        "answer_baseline",
        "answer_hybrid",
        "contexts_baseline",
        "contexts_hybrid",
        "ground_truth_baseline",
    ]
]

question  \
0                       Do I dream about houses?   
1                    Have I dreamt about whales?   
2                      Find my dreams about fire   
3  What recurring locations appear in my dreams?   
4            What does dreaming about fire mean?   
5            Do I have dreams featuring a clock?   
6                When have I dreamt about water?   
7          Find occurences of train in my dreams   
8                what should I cook for dinner?    
9                                 stupid chatbot   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               answer_baseline  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Yes, you have dreamt about houses.\n\n- **Dream 31 – 2024‑02‑03**  \n  *Excerpt:* “I arrived at a small wooden house on a quiet residential street… We went inside to visit an elderly woman who lived there. The house felt tidy and modest, with soft yellow lighting in the hallway.”\n\n- **Dream 15 – 2023‑08‑08**  \n  *Excerpt:* “I was in a house I used to rent years ago. It looked exactly the same… I sat down at the table… I felt very settled and also aware that I no longer lived there.”\n\nWould you like to look for any other specific themes or objects in your dreams?   
1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             I didn’t find any dreams featuring whales in your archive. Would you like to search for another theme or object?   
2                                                                                                                                                    

In [21]:
import json
from datetime import datetime

out_dir = Path("evals/results")
out_dir.mkdir(parents=True, exist_ok=True)

# Per-question outputs for traceability
baseline_df.to_csv(out_dir / "baseline_per_question_run_today.csv", index=False)
hybrid_df.to_csv(out_dir / "hybrid_per_question_run_today.csv", index=False)
print("Saved per-question CSVs")

# Summary metric scores -> JSON
def results_to_dict(ragas_result):
    df = ragas_result.to_pandas()
    numeric_cols = df.select_dtypes("number").columns
    return {col: round(float(df[col].mean()), 4) for col in numeric_cols}

summary = {
    "note": "RAGAS evaluation is non-deterministic (LLM-as-judge). Scores vary between runs.",
    "generated_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "metrics": [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
        "noise_sensitivity_relevant",
    ],
    "baseline": results_to_dict(baseline_results),
    "hybrid": results_to_dict(hybrid_results),
}

with open(out_dir / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Deliverable table for reporting
metrics_for_table = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
    "noise_sensitivity_relevant",
]

rows = []
for m in metrics_for_table:
    b = summary["baseline"].get(m)
    h = summary["hybrid"].get(m)
    delta = round(h - b, 4)
    rows.append({
        "metric": m,
        "baseline": b,
        "hybrid": h,
        "delta_hybrid_minus_baseline": delta,
    })

results_table = pd.DataFrame(rows)
results_table.to_csv(out_dir / "metrics_comparison_table.csv", index=False)
print("Saved metrics summary JSON and comparison table CSV")

# Simple metric interpretation (higher is better for all selected metrics)
improved = results_table[results_table["delta_hybrid_minus_baseline"] > 0]["metric"].tolist()
regressed = results_table[results_table["delta_hybrid_minus_baseline"] < 0]["metric"].tolist()
unchanged = results_table[results_table["delta_hybrid_minus_baseline"] == 0]["metric"].tolist()

print("\nConclusions")
print("- Improved metrics:", improved)
print("- Regressed metrics:", regressed)
print("- Unchanged metrics:", unchanged)

results_table

Saved per-question CSVs
Saved metrics summary JSON
{'baseline': {'faithfulness': 0.4944, 'answer_relevancy': 0.1931, 'context_precision': 0.5, 'context_recall': 0.35, 'noise_sensitivity_relevant': 0.4611}, 'hybrid': {'faithfulness': 0.5467, 'answer_relevancy': 0.3703, 'context_precision': 0.5, 'context_recall': 0.35, 'noise_sensitivity_relevant': 0.4271}}


## 4. Where We Stand (Submission Checklist)

- Task 5 baseline evaluation is implemented with a golden test set and RAGAS metrics (`faithfulness`, `context_precision`, `context_recall`) plus extra metrics (`answer_relevancy`, `noise_sensitivity_relevant`).
- Task 6 retriever upgrade is implemented and evaluated: baseline semantic-only pipeline vs upgraded hybrid retriever.
- Reporting artifacts are generated to `evals/results/`:
  - `baseline_per_question_run_today.csv`
  - `hybrid_per_question_run_today.csv`
  - `metrics_summary.json`
  - `metrics_comparison_table.csv`
- Because RAGAS uses an evaluator LLM, scores are non-deterministic; treat one run as a measurement sample and compare trends across repeated runs for stronger conclusions.